# 02 — Statistika Deskriptif Berkelompok
**PSAJ Statistika | Bab 4.1 & 4.2 Laporan**

Notebook ini menghasilkan:
- Nilai Min, Max, Range
- Aturan Sturges (Jumlah & Panjang Kelas)
- Tabel Distribusi Frekuensi Kelompok
- **Mean, Median, Modus** berkelompok (step-by-step)

---
**Input:** `outputs/merged_2025.csv`

In [1]:
import pandas as pd
import numpy as np
import math

# Load data bersih hasil notebook 01
df = pd.read_csv('outputs/merged_2025.csv')
X = df['SMA_Plus_Pct'].values   # Variabel X: Persentase Pendidikan SMA+
n = len(X)

print(f'Jumlah data (n): {n}')
print(df[['Provinsi', 'SMA_Plus_Pct', 'TPT_Pct']].to_string())

Jumlah data (n): 38
                Provinsi  SMA_Plus_Pct  TPT_Pct
0                   ACEH        51.440     5.64
1         SUMATERA UTARA        53.825     5.32
2         SUMATERA BARAT        48.390     5.62
3                   RIAU        45.550     4.16
4                  JAMBI        39.510     4.26
5       SUMATERA SELATAN        39.315     3.69
6               BENGKULU        43.400     3.41
7                LAMPUNG        34.455     4.21
8   KEP. BANGKA BELITUNG        40.630     4.45
9              KEP. RIAU        62.740     6.45
10           DKI JAKARTA        68.295     6.05
11            JAWA BARAT        40.670     6.77
12           JAWA TENGAH        30.975     4.66
13         DI YOGYAKARTA        54.930     3.46
14            JAWA TIMUR        35.420     3.88
15                BANTEN        45.165     6.69
16                  BALI        50.235     1.49
17   NUSA TENGGARA BARAT        36.380     3.06
18   NUSA TENGGARA TIMUR        34.410     3.31
19      KALIMANTAN B

## 4.1 — Eksplorasi Data Awal

In [2]:
# =============================================
# NILAI MINIMUM, MAKSIMUM, DAN JANGKAUAN (RANGE)
# =============================================
x_min  = X.min()
x_max  = X.max()
range_ = x_max - x_min

print('=== EKSPLORASI DATA AWAL ===')
print(f'Provinsi dengan SMA+ terendah : {df.loc[X.argmin(), "Provinsi"]} ({x_min:.2f}%)')
print(f'Provinsi dengan SMA+ tertinggi: {df.loc[X.argmax(), "Provinsi"]} ({x_max:.2f}%)')
print(f'Nilai Minimum (Xmin)           : {x_min:.2f}')
print(f'Nilai Maksimum (Xmax)          : {x_max:.2f}')
print(f'Jangkauan/Range (R = Xmax-Xmin): {range_:.2f}')

=== EKSPLORASI DATA AWAL ===
Provinsi dengan SMA+ terendah : PAPUA PEGUNUNGAN (16.66%)
Provinsi dengan SMA+ tertinggi: DKI JAKARTA (68.30%)
Nilai Minimum (Xmin)           : 16.66
Nilai Maksimum (Xmax)          : 68.30
Jangkauan/Range (R = Xmax-Xmin): 51.64


## 4.2 — Tabel Distribusi Frekuensi & Pemusatan Data

In [3]:
# =============================================
# ATURAN STURGES
# k = 1 + 3.3 * log10(n)
# =============================================
k_float  = 1 + 3.3 * math.log10(n)
k        = math.ceil(k_float)          # Jumlah kelas (dibulatkan ke atas)
p_float  = range_ / k
p        = math.ceil(p_float)          # Panjang kelas (dibulatkan ke atas)

print('=== ATURAN STURGES ===')
print(f'k = 1 + 3.3 × log₁₀({n})')
print(f'k = 1 + 3.3 × {math.log10(n):.4f}')
print(f'k = {k_float:.4f}  →  dibulatkan menjadi  k = {k} kelas')
print(f'\nPanjang Kelas (p = R/k):')
print(f'p = {range_:.2f} / {k} = {p_float:.4f}  →  dibulatkan menjadi  p = {p}')

=== ATURAN STURGES ===
k = 1 + 3.3 × log₁₀(38)
k = 1 + 3.3 × 1.5798
k = 6.2133  →  dibulatkan menjadi  k = 7 kelas

Panjang Kelas (p = R/k):
p = 51.64 / 7 = 7.3764  →  dibulatkan menjadi  p = 8


In [4]:
# =============================================
# TABEL DISTRIBUSI FREKUENSI KELOMPOK
# =============================================

# Batas bawah kelas pertama = x_min (atau dibulatkan ke bawah)
batas_bawah_awal = math.floor(x_min)

# Buat bins (tepi kelas)
bins = [batas_bawah_awal + i * p for i in range(k + 1)]
labels = [f'{bins[i]:.1f} – {bins[i+1]:.1f}' for i in range(k)]

# Hitung frekuensi
freq, _ = np.histogram(X, bins=bins)

# Tepi bawah (Tb) dan tepi atas (Ta)
tepi_bawah = [bins[i] - 0.5 for i in range(k)]
tepi_atas  = [bins[i+1] - 0.5 for i in range(k)]

# Titik tengah (xi)
titik_tengah = [(tepi_bawah[i] + tepi_atas[i]) / 2 for i in range(k)]

# Frekuensi kumulatif kurang dari dan lebih dari
fk_kurang = np.cumsum(freq)
fk_lebih  = n - np.cumsum(freq) + freq  # frekuensi kumulatif lebih dari

# f × xi
fi_xi = freq * np.array(titik_tengah)

# Susun tabel
tabel = pd.DataFrame({
    'Interval Kelas': labels,
    'Tepi Bawah (Tb)': tepi_bawah,
    'Tepi Atas (Ta)' : tepi_atas,
    'Titik Tengah (xi)': [round(t, 2) for t in titik_tengah],
    'Frekuensi (fi)' : freq,
    'fi × xi'        : [round(v, 2) for v in fi_xi],
    'fk < (kurang dari)' : fk_kurang,
    'fk > (lebih dari)'  : fk_lebih,
})

# Baris total
total_row = pd.DataFrame([{
    'Interval Kelas': 'TOTAL',
    'Tepi Bawah (Tb)': '',
    'Tepi Atas (Ta)': '',
    'Titik Tengah (xi)': '',
    'Frekuensi (fi)': freq.sum(),
    'fi × xi': round(fi_xi.sum(), 2),
    'fk < (kurang dari)': '',
    'fk > (lebih dari)': '',
}])

tabel_full = pd.concat([tabel, total_row], ignore_index=True)
print('=== TABEL DISTRIBUSI FREKUENSI KELOMPOK ===')
print(tabel_full.to_string(index=False))

# Simpan ke CSV
tabel.to_csv('outputs/tables/tabel_frekuensi.csv', index=False)
print('\nTabel tersimpan ke outputs/tables/tabel_frekuensi.csv')

=== TABEL DISTRIBUSI FREKUENSI KELOMPOK ===
Interval Kelas Tepi Bawah (Tb) Tepi Atas (Ta) Titik Tengah (xi)  Frekuensi (fi)  fi × xi fk < (kurang dari) fk > (lebih dari)
   16.0 – 24.0            15.5           23.5              19.5               1     19.5                  1                38
   24.0 – 32.0            23.5           31.5              27.5               2     55.0                  3                37
   32.0 – 40.0            31.5           39.5              35.5              12    426.0                 15                35
   40.0 – 48.0            39.5           47.5              43.5              10    435.0                 25                23
   48.0 – 56.0            47.5           55.5              51.5               9    463.5                 34                13
   56.0 – 64.0            55.5           63.5              59.5               3    178.5                 37                 4
   64.0 – 72.0            63.5           71.5              67.5           

In [5]:
# =============================================
# MEAN BERKELOMPOK
# x̄ = Σ(fi × xi) / Σfi
# =============================================
sum_fi    = freq.sum()
sum_fi_xi = fi_xi.sum()

mean_kelompok = sum_fi_xi / sum_fi

print('=== MEAN BERKELOMPOK ===')
print(f'Σfi        = {sum_fi}')
print(f'Σ(fi × xi) = {sum_fi_xi:.2f}')
print(f'x̄ = Σ(fi × xi) / Σfi')
print(f'x̄ = {sum_fi_xi:.2f} / {sum_fi}')
print(f'x̄ = {mean_kelompok:.4f} ≈ {mean_kelompok:.2f}%')

=== MEAN BERKELOMPOK ===
Σfi        = 38
Σ(fi × xi) = 1645.00
x̄ = Σ(fi × xi) / Σfi
x̄ = 1645.00 / 38
x̄ = 43.2895 ≈ 43.29%


In [6]:
# =============================================
# MEDIAN BERKELOMPOK
# Me = Tb + ((n/2 - Fk) / f_me) × p
# =============================================

# Cari kelas median: kelas di mana fk_kurang pertama kali >= n/2
half_n     = n / 2
idx_median = np.searchsorted(fk_kurang, half_n)  # index kelas median

Tb_me  = tepi_bawah[idx_median]
Fk_me  = fk_kurang[idx_median - 1] if idx_median > 0 else 0  # fk sebelum kelas median
f_me   = freq[idx_median]

median_kelompok = Tb_me + ((half_n - Fk_me) / f_me) * p

print('=== MEDIAN BERKELOMPOK ===')
print(f'n/2                                        = {half_n}')
print(f'Kelas Median                               = Kelas {idx_median + 1} ({labels[idx_median]})')
print(f'Tepi Bawah Kelas Median (Tb)               = {Tb_me}')
print(f'Frekuensi Kumulatif sebelum kelas (Fk)     = {Fk_me}')
print(f'Frekuensi Kelas Median (f_me)              = {f_me}')
print(f'Panjang Kelas (p)                          = {p}')
print(f'\nMe = Tb + ((n/2 - Fk) / f_me) × p')
print(f'Me = {Tb_me} + (({half_n} - {Fk_me}) / {f_me}) × {p}')
print(f'Me = {Tb_me} + ({half_n - Fk_me} / {f_me}) × {p}')
print(f'Me = {Tb_me} + {(half_n - Fk_me) / f_me:.4f} × {p}')
print(f'Me = {median_kelompok:.4f} ≈ {median_kelompok:.2f}%')

=== MEDIAN BERKELOMPOK ===
n/2                                        = 19.0
Kelas Median                               = Kelas 4 (40.0 – 48.0)
Tepi Bawah Kelas Median (Tb)               = 39.5
Frekuensi Kumulatif sebelum kelas (Fk)     = 15
Frekuensi Kelas Median (f_me)              = 10
Panjang Kelas (p)                          = 8

Me = Tb + ((n/2 - Fk) / f_me) × p
Me = 39.5 + ((19.0 - 15) / 10) × 8
Me = 39.5 + (4.0 / 10) × 8
Me = 39.5 + 0.4000 × 8
Me = 42.7000 ≈ 42.70%


In [7]:
# =============================================
# MODUS BERKELOMPOK
# Mo = Tb + (d1 / (d1 + d2)) × p
# =============================================

# Kelas modus = kelas dengan frekuensi tertinggi
idx_modus = np.argmax(freq)
Tb_mo     = tepi_bawah[idx_modus]
f_mo      = freq[idx_modus]
d1        = f_mo - (freq[idx_modus - 1] if idx_modus > 0 else 0)
d2        = f_mo - (freq[idx_modus + 1] if idx_modus < k - 1 else 0)

modus_kelompok = Tb_mo + (d1 / (d1 + d2)) * p

print('=== MODUS BERKELOMPOK ===')
print(f'Kelas Modus (frekuensi tertinggi)          = Kelas {idx_modus + 1} ({labels[idx_modus]}), fi = {f_mo}')
print(f'Tepi Bawah Kelas Modus (Tb)                = {Tb_mo}')
print(f'd1 = f_mo - f_(mo-1)                       = {f_mo} - {freq[idx_modus - 1] if idx_modus > 0 else 0} = {d1}')
print(f'd2 = f_mo - f_(mo+1)                       = {f_mo} - {freq[idx_modus + 1] if idx_modus < k - 1 else 0} = {d2}')
print(f'\nMo = Tb + (d1 / (d1 + d2)) × p')
print(f'Mo = {Tb_mo} + ({d1} / ({d1} + {d2})) × {p}')
print(f'Mo = {Tb_mo} + ({d1} / {d1+d2}) × {p}')
print(f'Mo = {Tb_mo} + {d1/(d1+d2):.4f} × {p}')
print(f'Mo = {modus_kelompok:.4f} ≈ {modus_kelompok:.2f}%')

=== MODUS BERKELOMPOK ===
Kelas Modus (frekuensi tertinggi)          = Kelas 3 (32.0 – 40.0), fi = 12
Tepi Bawah Kelas Modus (Tb)                = 31.5
d1 = f_mo - f_(mo-1)                       = 12 - 2 = 10
d2 = f_mo - f_(mo+1)                       = 12 - 10 = 2

Mo = Tb + (d1 / (d1 + d2)) × p
Mo = 31.5 + (10 / (10 + 2)) × 8
Mo = 31.5 + (10 / 12) × 8
Mo = 31.5 + 0.8333 × 8
Mo = 38.1667 ≈ 38.17%


In [8]:
# =============================================
# RANGKUMAN HASIL (untuk Bab 4.2 laporan)
# =============================================
print('\n===== RANGKUMAN STATISTIKA DESKRIPTIF =====')
print(f'Nilai Minimum   : {x_min:.2f}%')
print(f'Nilai Maksimum  : {x_max:.2f}%')
print(f'Jangkauan (R)   : {range_:.2f}')
print(f'Jumlah Kelas (k): {k}')
print(f'Panjang Kelas(p): {p}')
print(f'Mean  (x̄)      : {mean_kelompok:.2f}%')
print(f'Median (Me)     : {median_kelompok:.2f}%')
print(f'Modus  (Mo)     : {modus_kelompok:.2f}%')

# Simpan rangkuman
rangkuman = pd.DataFrame([{
    'Xmin': x_min, 'Xmax': x_max, 'Range': range_,
    'k (kelas)': k, 'p (panjang kelas)': p,
    'Mean': round(mean_kelompok, 4),
    'Median': round(median_kelompok, 4),
    'Modus': round(modus_kelompok, 4)
}])
rangkuman.to_csv('outputs/tables/rangkuman_deskriptif.csv', index=False)
print('\nRangkuman tersimpan ke outputs/tables/rangkuman_deskriptif.csv')


===== RANGKUMAN STATISTIKA DESKRIPTIF =====
Nilai Minimum   : 16.66%
Nilai Maksimum  : 68.30%
Jangkauan (R)   : 51.64
Jumlah Kelas (k): 7
Panjang Kelas(p): 8
Mean  (x̄)      : 43.29%
Median (Me)     : 42.70%
Modus  (Mo)     : 38.17%

Rangkuman tersimpan ke outputs/tables/rangkuman_deskriptif.csv
